In [12]:
!pip install -q -U accelerate peft bitsandbytes transformers trl datasets

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig , PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from trl import SFTTrainer, SFTConfig

In [14]:
model_name = "NousResearch/llama-2-7b-chat-hf"
new_model_name = "Llama-2-7b-chat-finetune"

lora_rank = 64
lora_alpha = 16
lora_dropout = 0.1

epochs = 1
train_batch_size = 4
lr = 2e-4

bf16 = torch.cuda.is_bf16_supported()

In [15]:
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
dataset = dataset.shuffle(seed=42).select(range(1000))
dataset = dataset.train_test_split(test_size=0.1, seed=42)

In [16]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model.gradient_checkpointing_enable()  

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# LoRA
peft_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias="none",
    task_type="CAUSAL_LM",
)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
training_args = SFTConfig(
    output_dir="./results",
    num_train_epochs=epochs,
    per_device_train_batch_size=train_batch_size,
    optim="paged_adamw_32bit",
    learning_rate=lr,
    bf16=bf16,
    fp16=not bf16,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    report_to="tensorboard",
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()
trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained("/content/Llama-2-7b-chat-finetune")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,1.258193,1.276979,1.142710,66218.000000,0.698989
100,1.298695,1.238577,1.167153,134892.000000,0.703588
150,1.262360,1.229679,1.217569,204019.000000,0.703360
200,1.472196,1.226647,1.183488,274738.000000,0.705156
225,1.260710,1.226471,1.187505,309798.000000,0.705143


In [26]:
logging.set_verbosity(logging.CRITICAL)

prompt = "What is the capital of France?"
formatted = f"<s>[INST] {prompt} [/INST]"

prompt2 = "What is the capital of Germany?"
formatted2 = f"<s>[INST] {prompt2} [/INST]"

pipe = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
    return_full_text=False,
)

response = pipe(formatted)
print(response[0]["generated_text"])

response2 = pipe(formatted2)
print(response2[0]["generated_text"])

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


The capital of France is Paris.
The capital of Germany is Berlin. 


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "NousResearch/llama-2-7b-chat-hf",
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model)
tokenizer = AutoTokenizer.from_pretrained()